# 35 — Ablation studies

Cuantifica el valor incremental de familias de features (`contract`, `contract+historical`, `all`) con un protocolo fijo sobre TRAIN/VALIDATION. Es diagnóstico: no modifica la selección congelada de 33.

In [1]:
from argos.common.paths import ProjectPaths
from argos.config.gold import load_gold_spec
from argos.config.modeling import load_modeling_spec

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()

m3_gate = paths.docs / "decisions" / "M3_GATE.md"
assert m3_gate.exists() and "M3 ✅ CLOSED" in m3_gate.read_text(encoding="utf-8"), (
    "M3 no está formalmente cerrado. No iniciar Modeling."
)
GOLD_SPEC = load_gold_spec(paths.configs / "gold.toml")
MODEL_SPEC = load_modeling_spec(paths.configs / "modeling.toml")
MODEL_DIR = paths.root / "artifacts" / "models" / "m4"
MANIFEST_DIR = paths.root / "artifacts" / "manifests"

{
    "gold_schema": GOLD_SPEC.schema_version,
    "modeling_schema": MODEL_SPEC.schema_version,
    "target": MODEL_SPEC.target,
    "model_matrix": MODEL_SPEC.model_matrix,
    "operational_review_fraction": MODEL_SPEC.ranking.operational_fraction,
}

assert (paths.reports / "modeling" / "34_validation.parquet").exists(), "Ejecuta 34 primero."
from argos.models.ablation import BUILD_IMPLEMENTATION
BUILD_IMPLEMENTATION


'm4-35-feature-family-ablation-v1'

In [2]:
from argos.models.ablation import build_ablation_study

result = build_ablation_study(
    root=paths.root,
    gold_spec=GOLD_SPEC,
    modeling_spec=MODEL_SPEC,
    reports_root=paths.reports,
    manifest_dir=MANIFEST_DIR,
)
result.metrics


,model,split,rows,positives,prevalence,pr_auc,roc_auc,log_loss,brier,ece,feature_family,feature_count
0,ablation_contract,validation_selection,428076,1568,0.003663,0.032450,0.845265,0.147740,0.037800,0.108043,contract,23
1,ablation_contract_historical,validation_selection,428076,1568,0.003663,0.033984,0.876469,0.132178,0.032476,0.094574,contract_historical,35
2,ablation_all,validation_selection,428076,1568,0.003663,0.030481,0.869739,0.146666,0.036863,0.102940,all,41


In [3]:
result.ranking

,model,split,review_fraction,reviewed,captured_positives,precision_at_k,recall_at_k,lift_at_k,cumulative_gain,feature_family,feature_count
0,ablation_contract,validation_selection,0.001,429,45,0.104895,0.028699,28.637166,0.028699,contract,23
1,ablation_contract,validation_selection,0.005,2141,99,0.046240,0.063138,12.623894,0.063138,contract,23
2,ablation_contract,validation_selection,0.010,4281,286,0.066807,0.182398,18.238773,0.182398,contract,23
3,ablation_contract,validation_selection,0.025,10702,476,0.044478,0.303571,12.142744,0.303571,contract,23
4,ablation_contract,validation_selection,0.050,21404,659,0.030789,0.420281,8.405534,0.420281,contract,23
5,ablation_contract,validation_selection,0.100,42808,890,0.020791,0.567602,5.675967,0.567602,contract,23
6,ablation_contract_historical,validation_selection,0.001,429,6,0.013986,0.003827,3.818289,0.003827,contract_historical,35
7,ablation_contract_historical,validation_selection,0.005,2141,73,0.034096,0.046556,9.308528,0.046556,contract_historical,35
8,ablation_contract_historical,validation_selection,0.010,4281,265,0.061901,0.169005,16.899563,0.169005,contract_historical,35
9,ablation_contract_historical,validation_selection,0.025,10702,539,0.050364,0.343750,13.749872,0.343750,contract_historical,35


In [4]:
result.validation

,check_id,category,blocking,passed,observed,requirement
0,ablation_families_complete,ablation,True,True,"{""expected"": [""all"", ""contract"", ""contract_his...",all predeclared feature-family ablations are e...
1,ablation_population_nonempty,evaluation,True,True,"{""positives"": 1568, ""rows"": 428076}",ablations are evaluated on the frozen validati...
2,ablation_metrics_finite,quality,True,True,"{""rows"": 3}",all ablation metrics are finite
3,ablation_test_isolation,test_isolation,True,True,"{""consumed_splits"": [""train"", ""validation_sele...",notebook 35 does not consume TEST rows and doe...


In [5]:
blocking = result.validation.loc[result.validation["blocking"]]
print("35 ✅ PASS" if result.ready_for_next_notebook else "35 ❌ FAIL")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
assert result.ready_for_next_notebook


35 ✅ PASS
Blocking checks: 4/4
